<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">Module 14 · Lab 14</div><div style="color:#17212b;font-size:30px;font-weight:750">Resource isolation and query protection for dashboards</div><p style="color:#475569;line-height:1.7">Separate dashboard and ad hoc work into two Workload Groups. Use multiple sessions to observe execution, queuing, rejection, and bypass; compare timeouts, reject excessive scans, then revoke and clean up.</p></div>

## What You Will Do

- Confirm fallback routing to normal.
- Create dashboard and ad hoc groups; grant dashboard-group USAGE_PRIV to the dedicated role.
- Observe slow queries running, waiting, failing, or bypassing the queue.
- Distinguish queue_timeout and query_timeout errors.
- Reject an unrestricted scan before execution with a SQL Block Rule.
- Revoke grants, remove unused objects, and verify cleanup.

## Prerequisites and Scope

**Prerequisites:** The course sandbox connection; no Level 1 business tables are required.

**Scope:** orders_monthly_l3, course_dashboard_l3, course_adhoc_l3, course_dashboard_reader_l3, and course_scan_guard_l3.

The Lab uses root and leaves normal and account properties unchanged. The global rule is created and removed within one cell using finally. If that cell is interrupted or its kernel restarts, rerun step 1 to remove the leftover rule.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.workload import BackgroundQuery, group_load, run_query, show_outcomes, wait_for_group

lab = connect_sandbox()

DASHBOARD = "course_dashboard_l3"
ADHOC = "course_adhoc_l3"

## 0. Which Group Does the Current Query Use?

This Lab has no query-level SET_VAR routing hint. Its routing precedence is therefore session workload_group, account default_workload_group, then normal.

Confirm that the session value is empty and the account default is normal. Save normal's configuration to check it again after the Lab.

In [ ]:
lab.sql("SELECT @@workload_group AS session_workload_group", title="Session workload_group")
lab.sql("SHOW PROPERTY LIKE 'default_workload_group'", title="Account default group")
lab.sql("SHOW WORKLOAD GROUPS", title="Existing Workload Groups")

NORMAL_BEFORE = lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'")

**Routing acceptance:** An empty session value and account default normal route this connection to normal. If root's default was changed earlier, the check fails. Restore the expected default before continuing.

In [ ]:
expect(lab.query("SELECT @@workload_group"), [("",)], title="Session group is empty")
expect(lab.query("SHOW PROPERTY LIKE 'default_workload_group'"), [("default_workload_group", "normal")], title="Account default group is normal")

## 1. Prepare the Table and Remove Leftover Objects

Workload Groups, roles, and SQL Block Rules are cluster-wide; dropping the Lab database does not remove them.

Remove leftovers in rule, role, then group order. Rebuild orders_monthly_l3 with the Module 13 monthly boundaries and one order per month.

In [ ]:
lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")

lab.execute("DROP TABLE IF EXISTS orders_monthly_l3")
lab.execute("""
CREATE TABLE orders_monthly_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01"),
    PARTITION p202503 VALUES LESS THAN ("2025-04-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("orders_monthly_l3", ["order_date", "order_id", "amount"], [("2025-01-15", 140001, 100), ("2025-02-15", 140002, 200), ("2025-03-15", 140003, 300)])
lab.sql("SELECT * FROM orders_monthly_l3 ORDER BY order_date, order_id", title="Monthly partitioned Lab table");

**Initial acceptance:** Expect three rows in p202501, p202502, and p202503. Step 3's slow query reads only March's row; step 5's unrestricted query scans all three partitions.

In [ ]:
expect(
    lab.query("""
SELECT CAST(order_date AS STRING), order_id, amount
FROM orders_monthly_l3
ORDER BY order_date, order_id
"""),
    [
        ("2025-01-15", 140001, "100.00"),
        ("2025-02-15", 140002, "200.00"),
        ("2025-03-15", 140003, "300.00"),
    ],
    title="One order in each of three months",
)

## 2. Create Dashboard and Ad Hoc Groups

The settings match the runbook in Course section 14.5. Ad hoc concurrency and queue capacity are both 1, making queuing and rejection visible with a few requests.

The Backend (BE) has no CPU cgroup configured, so max_cpu_percent is not enforced in this sandbox. Grant only dashboard-group USAGE_PRIV to the course role.

In [ ]:
lab.execute("""
CREATE WORKLOAD GROUP course_dashboard_l3
PROPERTIES (
    "max_concurrency" = "8",
    "max_queue_size" = "20",
    "queue_timeout" = "10000",
    "max_memory_percent" = "50"
)
""")
lab.execute("""
CREATE WORKLOAD GROUP course_adhoc_l3
PROPERTIES (
    "max_concurrency" = "1",
    "max_queue_size" = "1",
    "queue_timeout" = "30000",
    "max_memory_percent" = "30",
    "max_cpu_percent" = "30"
)
""")
lab.execute("CREATE ROLE course_dashboard_reader_l3")
lab.execute("GRANT USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' TO ROLE 'course_dashboard_reader_l3'")
lab.sql("SHOW WORKLOAD GROUPS LIKE 'course%'", title="The two course Workload Groups");

**Group acceptance:** Concurrency, queue capacity, wait time, and memory settings match the design. The role holds Usage_priv only for the dashboard group.

In [ ]:
expect(
    lab.query("""
SELECT NAME, MAX_CONCURRENCY, MAX_QUEUE_SIZE, QUEUE_TIMEOUT, MAX_MEMORY_PERCENT
FROM information_schema.workload_groups
WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY NAME
"""),
    [
        ("course_adhoc_l3", 1, 1, 30000, "30%"),
        ("course_dashboard_l3", 8, 20, 10000, "50%"),
    ],
    title="Limits for both groups",
)
expect(
    lab.query("""
SELECT GRANTEE, WORKLOAD_GROUP_NAME, PRIVILEGE_TYPE
FROM information_schema.workload_group_privileges
WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY GRANTEE, WORKLOAD_GROUP_NAME
"""),
    [("course_dashboard_reader_l3", "course_dashboard_l3", "Usage_priv")],
    title="Course role has usage only on the dashboard group",
)

**Read the grant:** USAGE_PRIV permits group use; it does not route queries there automatically. Production business intelligence (BI) accounts also need default_workload_group. This Lab selects groups through the session variable rather than modifying accounts.

## 3. Submit Five Queries: Run, Queue, or Reject?

sleep() simulates a slow scan of March's single row, so sleep(6) takes about six seconds. Each query opens a separate session, disables SQL Cache, sets its session variables, then runs.

A cache hit would return a previous result without entering the queue, so disabling the cache is necessary for this experiment.

- A occupies the ad hoc group's only running slot.
- B queues behind A.
- C enters the same group after the queue is full and should be rejected.
- D uses the dashboard group's own slot.
- E uses the ad hoc group with bypass_workload_group enabled.

Before submitting each dependent request, wait for the expected running and waiting counts to avoid a timing race.

In [ ]:
SLOW = "SELECT order_id, sleep({seconds}) AS slow_scan FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'"

a = BackgroundQuery(lab, "A ad hoc", SLOW.format(seconds=6), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = BackgroundQuery(lab, "B ad hoc", SLOW.format(seconds=1), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=1)

lab.sql("""
SELECT q.QUERY_STATUS, g.NAME AS workload_group, q.QUEUE_START_TIME, q.QUEUE_END_TIME
FROM information_schema.active_queries q
JOIN information_schema.workload_groups g ON q.WORKLOAD_GROUP_ID = g.ID
WHERE g.NAME = 'course_adhoc_l3'
ORDER BY q.QUERY_STATUS
""", title="A is running; B is queued");

In [ ]:
c = run_query(lab, "C ad hoc", SLOW.format(seconds=1), workload_group=ADHOC)
d = run_query(lab, "D dashboard", SLOW.format(seconds=1), workload_group=DASHBOARD)
e = run_query(lab, "E queue bypass", SLOW.format(seconds=1), workload_group=ADHOC, bypass_workload_group=True)
load_while_a_runs = group_load(lab, ADHOC)

a_outcome, b_outcome = a.result(), b.result()
show_outcomes("Outcomes of five queries", [a_outcome, b_outcome, c, d, e])
wait_for_group(lab, ADHOC, running=0, waiting=0);

**Queue acceptance:** A, B, D, and E succeed; C is rejected. When C, D, and E finish, the ordinary ad hoc counters still show one running and one waiting query. B takes longer than D because it waits for A first.

In [ ]:
expect(
    [(o.label, o.result) for o in (a_outcome, b_outcome, c, d, e)],
    [
        ("A ad hoc", "OK"),
        ("B ad hoc", "OK"),
        ("C ad hoc", "query waiting queue is full, queue capacity=1, waiting num=1"),
        ("D dashboard", "OK"),
        ("E queue bypass", "OK"),
    ],
    title="Execution, queuing, rejection, and bypass",
)
expect(load_while_a_runs, {"running": 1, "waiting": 1}, title="Ad hoc counters after C, D, and E finish")
expect(b_outcome.seconds > d.seconds + 3, True, title="B elapsed time includes queue wait")

**Read the results:** In active_queries, A is RUNNING and B is WAIT_IN_QUEUE. B has no QUEUE_END_TIME yet because it has not obtained a slot.

C is rejected because B occupies the only waiting position. D has a dashboard-group slot. E uses the same group as C but skips queuing through bypass_workload_group.

An ordinary user can set that session variable. Frontend (FE) queuing is therefore not an enforceable isolation boundary; group usage checks and BE limits still apply.

## 4. Limit Queue Wait and Execution Separately

queue_timeout limits waiting; query_timeout limits execution.

First shorten the ad hoc group's wait cap to one second and repeat the occupied-slot test. Then run F, a four-second query, with a two-second execution timeout. F does not select a group and uses normal.

This step changes the ad hoc wait cap. To repeat step 3 with its original settings, rerun from step 1.

In [ ]:
lab.execute('ALTER WORKLOAD GROUP course_adhoc_l3 PROPERTIES ("queue_timeout" = "1000")')

a = BackgroundQuery(lab, "A ad hoc", SLOW.format(seconds=3), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = run_query(lab, "B queue timeout", SLOW.format(seconds=1), workload_group=ADHOC)
f = run_query(lab, "F execution timeout", SLOW.format(seconds=4), query_timeout=2)

a_outcome = a.result()
show_outcomes("Queue and execution timeouts", [a_outcome, b, f])
wait_for_group(lab, ADHOC, running=0, waiting=0);

**Timeout acceptance:** A succeeds; B fails after roughly one second waiting with query queue timeout; F fails after roughly two seconds executing with query timeout.

In [ ]:
expect(
    [(o.label, o.result) for o in (a_outcome, b, f)],
    [
        ("A ad hoc", "OK"),
        ("B queue timeout", "query queue timeout, timeout: 1000 ms"),
        ("F execution timeout", "query timeout"),
    ],
    title="Two distinct timeout errors",
)
expect(b.seconds < 2, True, title="B stops waiting after about one second")
expect(1.5 <= f.seconds < 4, True, title="F is cancelled after about two seconds of execution")

**Read the results:** Queue timeout means the query never obtained its execution slot and did not run on BE. Execution timeout means it consumed resources before cancellation.

A user can change the session query_timeout. Production enforcement should use the administrator-controlled account property, which takes precedence over the session value.

## 5. Reject a Full Scan with a SQL Block Rule

The global rule permits scans of at most two partitions. An unrestricted query scans three and should fail; a March-only query prunes to one and succeeds.

The rule affects all sandbox sessions, including root. Create, test, and delete it within the same cell, using try/finally for cleanup. Interruptions or a kernel restart can still leave it behind; step 1 removes leftovers.

Use SUM(amount), because an unfiltered COUNT(*) may be answered without scanning partitions, as explained in Course section 14.4.

In [ ]:
lab.execute("""
CREATE SQL_BLOCK_RULE course_scan_guard_l3
PROPERTIES ("partition_num" = "2", "global" = "true", "enable" = "true")
""")
try:
    lab.sql("SHOW SQL_BLOCK_RULE FOR course_scan_guard_l3", title="Scan rule")
    full_scan = run_query(lab, "Full-table SUM", "SELECT SUM(amount) FROM orders_monthly_l3")
    one_month = run_query(lab, "March only", "SELECT SUM(amount) FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'")
finally:
    lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")

show_outcomes("Scan-rule outcomes", [full_scan, one_month])

**Rule acceptance:** The full scan hits course_scan_guard_l3, the one-month query succeeds, and the rule is absent after the cell completes.

In [ ]:
expect(
    [(o.label, o.result) for o in (full_scan, one_month)],
    [
        ("Full-table SUM", "sql hits sql block rule: course_scan_guard_l3, reach partition_num : 2"),
        ("March only", "OK"),
    ],
    title="Scan rule rejects the full scan",
)
expect(
    [row for row in lab.query("SHOW SQL_BLOCK_RULE") if row[0] == "course_scan_guard_l3"],
    [],
    title="Global rule has been removed",
)

**Read the results:** The rejected scan did not execute. Its error records the rule name and configured limit for troubleshooting.

partition_num : 2 is the limit, not the actual scan count. The March query prunes to one partition.

## 6. Revoke Access and Remove the Role and Groups

Check dependencies first: no Workload Policy references these groups, and this Lab has not made them any account's default.

Revoke USAGE_PRIV, remove the dedicated role, then drop both groups. Keep orders_monthly_l3 for further inspection; step 1 rebuilds it on the next run.

In [ ]:
expect(
    lab.query("""
SELECT COUNT(*) FROM information_schema.workload_policy
WHERE WORKLOAD_GROUP IN ('course_dashboard_l3', 'course_adhoc_l3')
"""),
    [(0,)],
    title="No Workload Policy references the course groups",
)
lab.execute("REVOKE USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' FROM ROLE 'course_dashboard_reader_l3'")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")
lab.sql("SHOW WORKLOAD GROUPS", title="Workload Groups after cleanup");

**Cleanup acceptance:** The groups, role, and usage grants are gone. Step 5 already removed the scan rule. normal matches the configuration saved in step 0.

In [ ]:
expect(
    lab.query("""
SELECT
    (SELECT COUNT(*) FROM information_schema.workload_groups
     WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')),
    (SELECT COUNT(*) FROM information_schema.workload_group_privileges
     WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3'))
"""),
    [(0, 0)],
    title="Course groups and usage grants have been removed",
)
expect([row[0] for row in lab.query("SHOW ROLES") if row[0] == "course_dashboard_reader_l3"], [], title="Course role has been removed")
expect(lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'"), NORMAL_BEFORE, title="normal remains unchanged")

## Independent Exercise: Plan Dashboard Resources

Five BI accounts submit about 20 simultaneous dashboard queries at peak refresh. Three analyst accounts submit irregular queries with unpredictable scope. The cluster has three FEs.

Write down:

1. Concurrency, queue capacity, and wait time for each group.
2. Usage grants and account routing.
3. Errors for a full queue, excessive waiting, and excessive execution.
4. A rule that rejects unrestricted scans and its account scope.
5. Rollback order.

Write your reasoning below before opening the reference.

In [ ]:
# TODO: Record both groups' settings, account routing, protection rules, and rollback order.

<details>
<summary>Reference Explanation</summary>

Queues are per FE. Size dashboard concurrency with the three FEs and verify connection distribution; reserve queue space for refresh bursts and allow brief waits. Keep ad hoc concurrency and waiting capacity small to bound accumulated work on BEs.

BI accounts need dashboard-group USAGE_PRIV and a default-group setting. Analysts need only the ad hoc group's usage privilege.

The errors are query waiting queue is full, query queue timeout, and query timeout. Enforce the production execution limit through the administrator-controlled account property.

Associate a partition_num SQL Block Rule with analyst accounts. A global rule also affects dashboards and administrators.

For rollback, remove policy dependencies, change account defaults, revoke usage privileges, remove unused roles and groups, remove obsolete rules, and verify with SHOW statements.

</details>

## Lab Review

- Routing precedence is query hint, session variable, account default, then normal.
- The Lab omits query-level routing hints.
- Concurrency, queue capacity, and wait limits control ordinary admission; the dashboard has its own slots.
- Ordinary users can bypass queuing; usage checks and BE limits remain.
- Queue timeout precedes execution, execution timeout cancels ongoing work, and scan rules reject before scanning.
- Revoke group grants before removal to prevent old grants from returning when a name is reused.
